# LeetCode 4 — Median of Two Sorted Arrays bằng Python

Notebook này giảng nhập môn bài toán **Hard** kinh điển:

```python
class Solution:
    def findMedianSortedArrays(self, nums1: List[int], nums2: List[int]) -> float:

```

Mục tiêu sau khi học xong:

1. Hiểu chính xác **median (trung vị)** là gì, cả trường hợp số phần tử lẻ và chẵn.
2. Viết được lời giải trực giác (gộp + sort), rồi cải tiến bằng **two pointers (hai con trỏ)** chỉ đi tới giữa.
3. Nắm **binary search (tìm kiếm nhị phân)** từ căn bản: `lo`, `hi`, `mid`, vì sao chỉ mất `O(log n)` bước.
4. Hiểu cách **tìm phần tử thứ k** bằng cách loại bỏ `k/2` phần tử mỗi bước.
5. Làm chủ cách chính: **binary search vách ngăn (partition)** trên mảng ngắn hơn, với `maxLeft`, `minRight` và `±inf`.
6. Tự kiểm thử, nhận ra lỗi hay gặp, và tự viết lại lời giải.

## 1. Đề bài

Cho hai mảng **đã sắp xếp tăng dần** `nums1` và `nums2` có kích thước lần lượt là `m` và `n`. Hãy trả về **median** của hai mảng đã sắp xếp đó.

Độ phức tạp thời gian tổng thể phải là `O(log (m+n))`.

**Ví dụ 1**

```text
Input: nums1 = [1,3], nums2 = [2]
Output: 2.00000
Explanation: merged array = [1,2,3] and median is 2.
```

**Ví dụ 2**

```text
Input: nums1 = [1,2], nums2 = [3,4]
Output: 2.50000
Explanation: merged array = [1,2,3,4] and median is (2 + 3) / 2 = 2.5.
```

**Constraints (ràng buộc)**

- `nums1.length == m`
- `nums2.length == n`
- `0 <= m <= 1000`
- `0 <= n <= 1000`
- `1 <= m + n <= 2000`
- `-10^6 <= nums1[i], nums2[i] <= 10^6`

Để ý 2 điều quan trọng từ constraints:

- Một trong hai mảng **có thể rỗng** (`m = 0` hoặc `n = 0`), nhưng không bao giờ cả hai cùng rỗng.
- Kiểu trả về là `float`, nên `2` phải hiểu là `2.0`, còn trung bình hai số như `(2 + 3) / 2` cho `2.5`.

## 2. Median (trung vị) là gì?

**Median** là giá trị "đứng giữa" khi ta xếp các số theo thứ tự tăng dần.

- Nếu có **lẻ** phần tử: median là phần tử chính giữa.
- Nếu có **chẵn** phần tử: median là **trung bình cộng** của 2 phần tử giữa.

| Dãy đã sắp xếp | Số phần tử | Phần tử giữa | Median |
|---|:---:|---|:---:|
| `[1, 2, 3]` | 3 (lẻ) | `2` | `2` |
| `[1, 2, 3, 4]` | 4 (chẵn) | `2` và `3` | `2.5` |
| `[5]` | 1 | `5` | `5` |
| `[-1, 4, 4, 10, 20]` | 5 | `4` | `4` |

Với `total` phần tử, đánh chỉ số từ `0` (index bắt đầu từ 0):

- `total` lẻ: median ở index `total // 2`.
- `total` chẵn: median là trung bình của index `total // 2 - 1` và `total // 2`.

Trong Python, `//` là **chia lấy phần nguyên** (`7 // 2 = 3`), còn `/` là **chia thực** luôn ra `float` (`7 / 2 = 3.5`).

In [1]:
def median_of_sorted_list(arr):
    total = len(arr)
    mid = total // 2
    if total % 2 == 1:          # lẻ
        return float(arr[mid])
    return (arr[mid - 1] + arr[mid]) / 2   # chẵn

for arr in [[1, 2, 3], [1, 2, 3, 4], [5], [-1, 4, 4, 10, 20]]:
    print(arr, "-> total =", len(arr), "| mid =", len(arr) // 2, "| median =", median_of_sorted_list(arr))

print(7 // 2, 7 / 2, (2 + 3) / 2)

[1, 2, 3] -> total = 3 | mid = 1 | median = 2.0
[1, 2, 3, 4] -> total = 4 | mid = 2 | median = 2.5
[5] -> total = 1 | mid = 0 | median = 5.0
[-1, 4, 4, 10, 20] -> total = 5 | mid = 2 | median = 4.0
3 3.5 2.5


### Lộ trình 4 cách giải

Ta sẽ đi từ dễ đến khó, mỗi bước trả lời "cách cũ vướng ở đâu?":

| # | Cách | Thời gian | Ghi chú |
|:---:|---|---|---|
| 1 | Gộp hai mảng rồi `sorted` | `O((m+n) log(m+n))` | Dễ nhất, chưa dùng tính chất "đã sắp xếp" |
| 2 | Merge bằng hai con trỏ, chỉ đi tới giữa | `O(m+n)` | Dùng được "đã sắp xếp", bộ nhớ `O(1)` |
| 3 | Tìm phần tử thứ `k`, loại `k/2` mỗi bước | `O(log(m+n))` | Đạt yêu cầu đề |
| 4 | Binary search vách ngăn trên mảng ngắn hơn | `O(log(min(m,n)))` | **Cách chính**, gọn và nhanh nhất |

Hai cách đầu giúp ta hiểu bài. Hai cách sau dùng **binary search**, nên ta sẽ học binary search từ căn bản ở mục 5.

## 3. Cách 1 — Gộp hai mảng rồi sắp xếp

Cách người mới nghĩ ra đầu tiên: đúng theo định nghĩa median.

1. Gộp hai mảng thành một: `nums1 + nums2`.
2. Sắp xếp lại: `sorted(...)`.
3. Lấy phần tử giữa (hoặc trung bình 2 phần tử giữa).

Hai công cụ Python:

- `a + b` với hai list sẽ tạo **list mới** gồm các phần tử của `a` rồi tới `b` (nối, không sắp xếp).
- `sorted(x)` trả về **list mới** đã sắp xếp tăng dần, không sửa `x`.

In [2]:
a = [1, 3]
b = [2]
merged = a + b
print("a + b      =", merged)
print("sorted(..) =", sorted(merged))
print("a vẫn là   =", a)

a + b      = [1, 3, 2]
sorted(..) = [1, 2, 3]
a vẫn là   = [1, 3]


In [3]:
def median_sort(nums1, nums2):
    merged = sorted(nums1 + nums2)
    return median_of_sorted_list(merged)

print(median_sort([1, 3], [2]))        # Ví dụ 1 -> 2.0
print(median_sort([1, 2], [3, 4]))     # Ví dụ 2 -> 2.5
print(median_sort([], [7]))            # một mảng rỗng -> 7.0

2.0
2.5
7.0


### Độ phức tạp của Cách 1

Đặt `m = len(nums1)`, `n = len(nums2)`, `N = m + n`.

- Thời gian: `O(N log N)` do sắp xếp.
- Bộ nhớ: `O(N)` cho list gộp.

**Vướng ở đâu?** Đề đã cho hai mảng **sắp xếp sẵn**, nhưng ta lại vứt thông tin đó đi và sắp xếp lại từ đầu. Đây là dấu hiệu có thể làm tốt hơn.

Cách này vẫn rất quý: nó là **brute force** (cách vét cạn, dễ đúng) để ta đối chiếu với các cách nhanh hơn ở phần test.

## 4. Cách 2 — Merge bằng hai con trỏ (two pointers)

### Hai con trỏ là gì?

**Two pointers** là kỹ thuật dùng 2 biến chỉ số, ở đây là `i` chạy trên `nums1` và `j` chạy trên `nums2`.

Vì cả hai mảng đã tăng dần, phần tử **nhỏ nhất chưa lấy** luôn là `nums1[i]` hoặc `nums2[j]`. Ta so sánh hai số đó, lấy số nhỏ hơn và tiến con trỏ tương ứng thêm 1.

Ví dụ merge `[1, 4, 7]` và `[2, 3, 9]`:

| Bước | `nums1[i]` | `nums2[j]` | Lấy | Kết quả tạm |
|:---:|:---:|:---:|:---:|---|
| 1 | 1 | 2 | 1 (từ nums1) | `[1]` |
| 2 | 4 | 2 | 2 (từ nums2) | `[1, 2]` |
| 3 | 4 | 3 | 3 (từ nums2) | `[1, 2, 3]` |
| 4 | 4 | 9 | 4 (từ nums1) | `[1, 2, 3, 4]` |
| 5 | 7 | 9 | 7 (từ nums1) | `[1, 2, 3, 4, 7]` |
| 6 | hết | 9 | 9 (phần còn lại) | `[1, 2, 3, 4, 7, 9]` |

Khi một mảng hết, phần còn lại của mảng kia đã sắp xếp sẵn, chỉ việc nối vào.

In [4]:
def merge_sorted(nums1, nums2):
    i, j = 0, 0
    result = []
    while i < len(nums1) and j < len(nums2):
        if nums1[i] <= nums2[j]:
            result.append(nums1[i]); i += 1
        else:
            result.append(nums2[j]); j += 1
    result.extend(nums1[i:])   # phần còn lại (nếu có)
    result.extend(nums2[j:])
    return result

print(merge_sorted([1, 4, 7], [2, 3, 9]))
print(merge_sorted([], [5, 6]))

[1, 2, 3, 4, 7, 9]
[5, 6]


In [5]:
def median_merge(nums1, nums2):
    return median_of_sorted_list(merge_sorted(nums1, nums2))

print(median_merge([1, 3], [2]))      # 2.0
print(median_merge([1, 2], [3, 4]))   # 2.5

2.0
2.5


### Cải tiến: chỉ đi tới giữa, bộ nhớ O(1)

Để tính median ta **không cần cả mảng gộp**. Ta chỉ cần phần tử ở index `total // 2` (và phần tử ngay trước nó khi `total` chẵn).

Ý tưởng:

- Vẫn dùng `i`, `j` như merge, nhưng **không lưu** list kết quả.
- Mỗi bước "lấy" ra một số, ta giữ 2 biến: `prev` (số lấy ở bước trước) và `cur` (số vừa lấy).
- Lấy đúng `total // 2 + 1` số rồi dừng. Khi đó `cur` là phần tử index `total // 2`, `prev` là phần tử index `total // 2 - 1`.

Điều kiện chọn lấy từ `nums1`:

```python
if j >= n or (i < m and nums1[i] <= nums2[j]):
```

Đọc là: "nếu `nums2` đã hết, **hoặc** `nums1` còn và số đầu của `nums1` không lớn hơn số đầu của `nums2`". Thứ tự kiểm tra quan trọng: phải chắc `i < m` trước khi đọc `nums1[i]`, nếu không sẽ `IndexError`.

In [6]:
def median_merge_half(nums1, nums2):
    m, n = len(nums1), len(nums2)
    total = m + n
    i = j = 0
    prev = cur = 0
    for _ in range(total // 2 + 1):
        prev = cur
        if j >= n or (i < m and nums1[i] <= nums2[j]):
            cur = nums1[i]; i += 1
        else:
            cur = nums2[j]; j += 1
    if total % 2 == 1:
        return float(cur)
    return (prev + cur) / 2

print(median_merge_half([1, 3], [2]))         # 2.0
print(median_merge_half([1, 2], [3, 4]))      # 2.5
print(median_merge_half([], [1, 2, 3, 4]))    # 2.5

2.0
2.5
2.5


### Trace Cách 2 — `nums1 = [1, 3]`, `nums2 = [2, 4, 5, 6]`

`total = 6` (chẵn), cần lấy `6 // 2 + 1 = 4` số. Mảng gộp là `[1, 2, 3, 4, 5, 6]`, median là `(3 + 4) / 2 = 3.5`.

| Bước | `i` | `j` | So sánh | Lấy từ | `prev` | `cur` |
|:---:|:---:|:---:|---|:---:|:---:|:---:|
| 0 | 0 | 0 | `1 <= 2` | nums1 | 0 | 1 |
| 1 | 1 | 0 | `3 <= 2` sai | nums2 | 1 | 2 |
| 2 | 1 | 1 | `3 <= 4` | nums1 | 2 | 3 |
| 3 | 2 | 1 | nums1 đã hết | nums2 | 3 | 4 |

Ở bước 3, `i = 2 = m` nên nhánh "nums1 hết" được dùng. Kết quả `(prev + cur) / 2 = (3 + 4) / 2 = 3.5`.

In [7]:
def trace_merge_half(nums1, nums2):
    m, n = len(nums1), len(nums2)
    total = m + n
    i = j = 0
    prev = cur = 0
    print(f"total = {total}, cần lấy {total // 2 + 1} số")
    for step in range(total // 2 + 1):
        prev = cur
        if j >= n or (i < m and nums1[i] <= nums2[j]):
            reason = "nums2 hết" if j >= n else f"{nums1[i]} <= {nums2[j]}"
            print(f"Bước {step}: i={i}, j={j}, {reason:>10} -> lấy nums1[{i}]={nums1[i]}")
            cur = nums1[i]; i += 1
        else:
            reason = "nums1 hết" if i >= m else f"{nums1[i]} > {nums2[j]}"
            print(f"Bước {step}: i={i}, j={j}, {reason:>10} -> lấy nums2[{j}]={nums2[j]}")
            cur = nums2[j]; j += 1
        print(f"         prev={prev}, cur={cur}")
    ans = float(cur) if total % 2 else (prev + cur) / 2
    print("Median =", ans)

trace_merge_half([1, 3], [2, 4, 5, 6])

total = 6, cần lấy 4 số
Bước 0: i=0, j=0,     1 <= 2 -> lấy nums1[0]=1
         prev=0, cur=1
Bước 1: i=1, j=0,      3 > 2 -> lấy nums2[0]=2
         prev=1, cur=2
Bước 2: i=1, j=1,     3 <= 4 -> lấy nums1[1]=3
         prev=2, cur=3
Bước 3: i=2, j=1,  nums1 hết -> lấy nums2[1]=4
         prev=3, cur=4
Median = 3.5


### Độ phức tạp của Cách 2

- Thời gian: `O(m + n)` — chính xác là khoảng `(m + n) / 2 + 1` bước.
- Bộ nhớ: `O(1)` với bản `median_merge_half` (bản `median_merge` dùng `O(m + n)`).

**Vướng ở đâu?** Mỗi bước ta chỉ loại được **1 phần tử**. Đề yêu cầu `O(log(m+n))`, tức là mỗi bước phải loại được **cả một nửa** số ứng viên. Công cụ làm được điều đó chính là **binary search**.

## 5. Nền tảng — Binary search (tìm kiếm nhị phân) từ căn bản

### Trò chơi đoán số

Bạn nghĩ một số từ 1 đến 100. Tôi đoán, bạn chỉ trả lời "lớn hơn" hay "nhỏ hơn".

- Đoán 50 → "nhỏ hơn" → chỉ còn 1..49.
- Đoán 25 → "lớn hơn" → chỉ còn 26..49.
- ...

Mỗi lần đoán **vào giữa**, ta loại được **một nửa** khả năng. Đó là binary search.

### Binary search trên mảng đã sắp xếp

Tìm `target` trong mảng tăng dần `arr`:

- `lo`, `hi`: biên trái, biên phải của **vùng còn nghi ngờ** (`arr[lo..hi]`).
- `mid = (lo + hi) // 2`: vị trí ở giữa.
- Nếu `arr[mid] == target` → tìm thấy.
- Nếu `arr[mid] < target` → target chỉ có thể ở bên phải → `lo = mid + 1`.
- Nếu `arr[mid] > target` → target chỉ có thể ở bên trái → `hi = mid - 1`.
- Lặp khi `lo <= hi`. Khi `lo > hi`, vùng nghi ngờ rỗng → không có.

Vì sao được phép bỏ một nửa? Vì mảng **đã sắp xếp**: nếu `arr[mid] < target` thì mọi phần tử bên trái `mid` còn nhỏ hơn nữa.

In [8]:
def binary_search(arr, target):
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        print(f"  lo={lo}, hi={hi}, mid={mid}, arr[mid]={arr[mid]}")
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

arr = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("Tìm 23:"); print("->", binary_search(arr, 23))
print("Tìm 7:");  print("->", binary_search(arr, 7))

Tìm 23:
  lo=0, hi=9, mid=4, arr[mid]=16
  lo=5, hi=9, mid=7, arr[mid]=56
  lo=5, hi=6, mid=5, arr[mid]=23
-> 5
Tìm 7:
  lo=0, hi=9, mid=4, arr[mid]=16
  lo=0, hi=3, mid=1, arr[mid]=5
  lo=2, hi=3, mid=2, arr[mid]=8
-> -1


### Vì sao binary search chỉ mất O(log n) bước?

Mỗi bước, độ dài vùng nghi ngờ giảm còn khoảng một nửa: `n → n/2 → n/4 → ... → 1`.

Số lần chia đôi tới khi còn 1 phần tử là khoảng `log₂ n` (logarit cơ số 2). Ví dụ `log₂ 1024 = 10` vì `2^10 = 1024`.

Nên dù mảng có 1 triệu phần tử, binary search cũng chỉ cần khoảng 20 bước.

In [9]:
def count_halving(n):
    steps = 0
    while n > 1:
        n //= 2
        steps += 1
    return steps

for n in [8, 1000, 2000, 1_000_000]:
    print(f"n = {n:>9} -> chia đôi {count_halving(n):>2} lần")

n =         8 -> chia đôi  3 lần
n =      1000 -> chia đôi  9 lần
n =      2000 -> chia đôi 10 lần
n =   1000000 -> chia đôi 19 lần


### Dạng tổng quát: tìm vị trí "chuyển trạng thái"

Binary search không chỉ để tìm một giá trị. Nó dùng được mỗi khi ta có một điều kiện **đơn điệu (monotonic)**: khi đi từ trái sang phải, kết quả kiểm tra chỉ đổi **một lần**, ví dụ:

```text
vị trí:     0      1      2     3     4
điều kiện: False  False  True  True  True
                         ^ cần tìm chỗ này
```

Ví dụ: "vị trí đầu tiên có `arr[i] >= x`" (gọi là **lower bound**). Ta xem giữa: nếu điều kiện đúng thì đáp án ở `mid` hoặc bên trái; nếu sai thì đáp án ở bên phải.

Python có sẵn module `bisect`: `bisect.bisect_left(arr, x)` trả về đúng lower bound này. Ta tự viết để hiểu, rồi so với `bisect`.

Ý tưởng "nhìn ở giữa → biết đáp án lệch về bên nào → bỏ nửa kia" chính là thứ ta dùng ở Cách 3 và Cách 4.

In [10]:
import bisect

def lower_bound(arr, x):
    lo, hi = 0, len(arr)          # đáp án có thể là len(arr) (mọi số đều < x)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] >= x:          # điều kiện đúng -> đáp án ở mid hoặc bên trái
            hi = mid
        else:                      # điều kiện sai -> đáp án ở bên phải
            lo = mid + 1
    return lo

arr = [1, 3, 3, 5, 8]
for x in [0, 3, 4, 8, 9]:
    print(f"x={x}: lower_bound={lower_bound(arr, x)}, bisect_left={bisect.bisect_left(arr, x)}")

x=0: lower_bound=0, bisect_left=0
x=3: lower_bound=1, bisect_left=1
x=4: lower_bound=3, bisect_left=3
x=8: lower_bound=4, bisect_left=4
x=9: lower_bound=5, bisect_left=5


## 6. Cách 3 — Tìm phần tử thứ k (loại bỏ k/2 mỗi bước)

### Đưa median về bài "phần tử nhỏ thứ k"

Gọi `kth(k)` là phần tử nhỏ thứ `k` (đếm từ **1**) trong hai mảng gộp lại. Với `total = m + n`:

- `total` lẻ: median `= kth((total + 1) // 2)`. Ví dụ `total = 5` → `kth(3)`.
- `total` chẵn: median `= (kth(total // 2) + kth(total // 2 + 1)) / 2`. Ví dụ `total = 4` → `kth(2)`, `kth(3)`.

### Mấu chốt: so sánh phần tử thứ k/2 của mỗi mảng

Giả sử cần `kth(k)` từ hai mảng `A`, `B`. Lấy `p = k // 2` phần tử đầu của `A` và `p` phần tử đầu của `B`, so sánh **phần tử cuối** của hai nhóm đó: `A[p-1]` và `B[p-1]`.

Nếu `A[p-1] <= B[p-1]`, thì `A[0..p-1]` **không thể chứa phần tử thứ k** (trừ khi giá trị bằng nhau). Lý do:

- Số phần tử có thể đứng trước `A[p-1]` trong mảng gộp: tối đa `p - 1` phần tử của `A` (đứng trước nó trong `A`) và tối đa `p - 1` phần tử của `B` (các phần tử trước `B[p-1]`, vì `B[p-1] >= A[p-1]`).
- Vậy `A[p-1]` đứng tối đa ở vị trí thứ `(p - 1) + (p - 1) + 1 = 2p - 1 <= k - 1`, tức là **trước** phần tử thứ `k`.
- Các phần tử trước `A[p-1]` trong `A` càng đứng sớm hơn.

→ Bỏ luôn `p` phần tử đầu của `A`, và bây giờ chỉ cần tìm `kth(k - p)` trong phần còn lại. Mỗi bước, `k` giảm khoảng một nửa!

Trường hợp `B[p-1] < A[p-1]` thì làm ngược lại: bỏ `p` phần tử đầu của `B`.

### Ba trường hợp dừng và một chi tiết biên

- Nếu `A` đã bị bỏ hết → đáp án là phần tử thứ `k` của phần còn lại của `B`.
- Nếu `B` đã bị bỏ hết → tương tự với `A`.
- Nếu `k == 1` → đáp án là `min(đầu A, đầu B)`.

**Chi tiết biên:** nếu một mảng còn ít hơn `k // 2` phần tử, ta chỉ lấy được những gì nó còn. Vì vậy số phần tử xét ở mỗi mảng là `min(k // 2, số phần tử còn lại)`. Lập luận ở trên vẫn đúng vì tổng số phần tử xét ở hai mảng vẫn `<= k`.

Để không phải cắt list (slicing tốn `O(độ dài)`), ta dùng **offset**: `i` là số phần tử đã bỏ ở `nums1`, `j` là số đã bỏ ở `nums2`. Phần còn lại là `nums1[i:]` và `nums2[j:]` nhưng không tạo list mới.

Thí nghiệm nhỏ: `A = [1, 3, 5, 7]`, `B = [2, 4, 6, 8, 10]`, tìm `kth(5)` (mảng gộp `[1,2,3,4,5,6,7,8,10]`, đáp án `5`).

In [11]:
A = [1, 3, 5, 7]
B = [2, 4, 6, 8, 10]
k = 5
p = k // 2
print("p =", p, "| A[p-1] =", A[p - 1], "| B[p-1] =", B[p - 1])
print("A[p-1] <= B[p-1] ->", A[p - 1] <= B[p - 1], "=> bỏ", A[:p], ", còn tìm kth", k - p)
print("Kiểm tra bằng sorted:", sorted(A + B)[k - 1], "==", sorted(A[p:] + B)[k - p - 1])

p = 2 | A[p-1] = 3 | B[p-1] = 4
A[p-1] <= B[p-1] -> True => bỏ [1, 3] , còn tìm kth 3
Kiểm tra bằng sorted: 5 == 5


In [12]:
def kth_smallest(nums1, nums2, k):
    """Phần tử nhỏ thứ k (k bắt đầu từ 1) của hai mảng đã sắp xếp."""
    m, n = len(nums1), len(nums2)
    i = j = 0                       # số phần tử đã loại ở mỗi mảng
    while True:
        if i == m:
            return nums2[j + k - 1]
        if j == n:
            return nums1[i + k - 1]
        if k == 1:
            return min(nums1[i], nums2[j])
        half = k // 2
        ni = min(i + half, m) - 1   # index phần tử cuối nhóm xét ở nums1
        nj = min(j + half, n) - 1   # index phần tử cuối nhóm xét ở nums2
        if nums1[ni] <= nums2[nj]:
            k -= ni - i + 1         # bỏ nums1[i..ni]
            i = ni + 1
        else:
            k -= nj - j + 1         # bỏ nums2[j..nj]
            j = nj + 1


def median_kth(nums1, nums2):
    total = len(nums1) + len(nums2)
    if total % 2 == 1:
        return float(kth_smallest(nums1, nums2, total // 2 + 1))
    left = kth_smallest(nums1, nums2, total // 2)
    right = kth_smallest(nums1, nums2, total // 2 + 1)
    return (left + right) / 2

print(kth_smallest([1, 3, 5, 7], [2, 4, 6, 8, 10], 5))   # 5
print(median_kth([1, 3], [2]))                            # 2.0
print(median_kth([1, 2], [3, 4]))                         # 2.5
print(median_kth([], [1]))                                # 1.0

5
2.0
2.5
1.0


### Trace Cách 3 — `nums1 = [1, 3, 5, 7]`, `nums2 = [2, 4, 6, 8, 10]`, `k = 5`

| Vòng | `i` | `j` | `k` | `half` | So sánh | Hành động |
|:---:|:---:|:---:|:---:|:---:|---|---|
| 1 | 0 | 0 | 5 | 2 | `nums1[1]=3` vs `nums2[1]=4` | `3 <= 4` → bỏ `[1, 3]`, `k = 3` |
| 2 | 2 | 0 | 3 | 1 | `nums1[2]=5` vs `nums2[0]=2` | `5 > 2` → bỏ `[2]`, `k = 2` |
| 3 | 2 | 1 | 2 | 1 | `nums1[2]=5` vs `nums2[1]=4` | `5 > 4` → bỏ `[4]`, `k = 1` |
| 4 | 2 | 2 | 1 | – | `k == 1` | trả `min(5, 6) = 5` |

`k` đi `5 → 3 → 2 → 1`: mỗi vòng `k` giảm khoảng một nửa, nên số vòng là `O(log k)`.

Cell dưới in đúng trạng thái này, sau đó thử một ca **mảng ngắn bị hết sớm** để thấy nhánh `min(..., m)`.

In [13]:
def trace_kth(nums1, nums2, k):
    m, n = len(nums1), len(nums2)
    i = j = 0
    step = 1
    print(f"Tìm kth({k}) của {nums1} và {nums2}")
    while True:
        if i == m:
            print(f"Vòng {step}: nums1 hết -> nums2[{j}+{k}-1] = {nums2[j + k - 1]}"); return nums2[j + k - 1]
        if j == n:
            print(f"Vòng {step}: nums2 hết -> nums1[{i}+{k}-1] = {nums1[i + k - 1]}"); return nums1[i + k - 1]
        if k == 1:
            print(f"Vòng {step}: k == 1 -> min({nums1[i]}, {nums2[j]})"); return min(nums1[i], nums2[j])
        half = k // 2
        ni, nj = min(i + half, m) - 1, min(j + half, n) - 1
        msg = f"Vòng {step}: i={i}, j={j}, k={k}, nums1[{ni}]={nums1[ni]} vs nums2[{nj}]={nums2[nj]}"
        if nums1[ni] <= nums2[nj]:
            print(msg, "-> bỏ", nums1[i:ni + 1]); k -= ni - i + 1; i = ni + 1
        else:
            print(msg, "-> bỏ", nums2[j:nj + 1]); k -= nj - j + 1; j = nj + 1
        step += 1

print("=>", trace_kth([1, 3, 5, 7], [2, 4, 6, 8, 10], 5))
print()
print("=>", trace_kth([1], [2, 3, 4, 5, 6, 7], 6))

Tìm kth(5) của [1, 3, 5, 7] và [2, 4, 6, 8, 10]
Vòng 1: i=0, j=0, k=5, nums1[1]=3 vs nums2[1]=4 -> bỏ [1, 3]
Vòng 2: i=2, j=0, k=3, nums1[2]=5 vs nums2[0]=2 -> bỏ [2]
Vòng 3: i=2, j=1, k=2, nums1[2]=5 vs nums2[1]=4 -> bỏ [4]
Vòng 4: k == 1 -> min(5, 6)
=> 5

Tìm kth(6) của [1] và [2, 3, 4, 5, 6, 7]
Vòng 1: i=0, j=0, k=6, nums1[0]=1 vs nums2[2]=4 -> bỏ [1]
Vòng 2: nums1 hết -> nums2[0+5-1] = 6
=> 6


### Độ phức tạp của Cách 3

- Thời gian: mỗi vòng loại khoảng `k / 2` phần tử, nên một lần gọi `kth_smallest` tốn `O(log k)`, với `k <= m + n`. Gọi tối đa 2 lần → `O(log(m + n))`. **Đạt yêu cầu đề.**
- Bộ nhớ: `O(1)` vì dùng offset `i`, `j` thay vì cắt list.

**Vướng ở đâu?** Code có nhiều chỗ dễ sai biên: `k` đếm từ 1, `ni - i + 1`, `min(i + half, m)`, phải gọi 2 lần khi chẵn. Ta sẽ đổi góc nhìn: thay vì "đi tìm phần tử thứ k", hãy **tìm chỗ cắt đôi** hai mảng. Đó là Cách 4.

## 7. Cách 4 (cách chính) — Binary search "vách ngăn" (partition)

### Median chia mảng gộp thành hai nửa

Nhìn lại median theo một cách khác: median là chỗ **cắt** mảng gộp thành 2 phần sao cho:

1. **Nửa trái** có số phần tử bằng nửa phải (hoặc **nhiều hơn 1** nếu tổng lẻ).
2. **Mọi** phần tử nửa trái `<=` **mọi** phần tử nửa phải.

```text
total = 7:  [1, 2, 3, 4 | 5, 6, 7]   nửa trái có 4 = (7 + 1) // 2 phần tử
            median = max(nửa trái) = 4

total = 6:  [1, 2, 3 | 4, 5, 6]      nửa trái có 3 = (6 + 1) // 2 phần tử
            median = (max(nửa trái) + min(nửa phải)) / 2 = (3 + 4) / 2
```

Ta gọi dấu `|` là **vách ngăn**. Đặt `half = (m + n + 1) // 2` là số phần tử của nửa trái (công thức này đúng cho cả lẻ và chẵn).

### Vách ngăn trên từng mảng

Không cần gộp! Nửa trái của mảng gộp chính là **một đoạn đầu của `nums1`** cộng **một đoạn đầu của `nums2`**. Vậy ta đặt một vách ngăn trên mỗi mảng:

```text
nums1:  nums1[0] ... nums1[i-1]  |  nums1[i] ... nums1[m-1]
nums2:  nums2[0] ... nums2[j-1]  |  nums2[j] ... nums2[n-1]
        \______ nửa trái ______/    \______ nửa phải ______/
```

- `i` = số phần tử của `nums1` đứng bên trái vách (`0 <= i <= m`).
- `j` = số phần tử của `nums2` đứng bên trái vách.
- Nửa trái phải có đúng `half` phần tử → **`j = half - i`**.

Vậy chỉ cần chọn `i`, còn `j` tự suy ra. Bài toán chỉ còn: **tìm `i` đúng**.

Ta đặt tên 4 số nằm sát vách:

| Tên | Giá trị | Ý nghĩa |
|---|---|---|
| `maxLeftA` | `nums1[i-1]` | số lớn nhất của nums1 bên trái vách |
| `minRightA` | `nums1[i]` | số nhỏ nhất của nums1 bên phải vách |
| `maxLeftB` | `nums2[j-1]` | số lớn nhất của nums2 bên trái vách |
| `minRightB` | `nums2[j]` | số nhỏ nhất của nums2 bên phải vách |

In [14]:
A = [1, 3, 8, 9, 15]
B = [7, 11, 18, 19, 21, 25]
m, n = len(A), len(B)
half = (m + n + 1) // 2
print("m =", m, "n =", n, "half =", half)
for i in range(m + 1):
    j = half - i
    print(f"i={i}, j={j}: trái = {A[:i]} + {B[:j]} | phải = {A[i:]} + {B[j:]}")

m = 5 n = 6 half = 6
i=0, j=6: trái = [] + [7, 11, 18, 19, 21, 25] | phải = [1, 3, 8, 9, 15] + []
i=1, j=5: trái = [1] + [7, 11, 18, 19, 21] | phải = [3, 8, 9, 15] + [25]
i=2, j=4: trái = [1, 3] + [7, 11, 18, 19] | phải = [8, 9, 15] + [21, 25]
i=3, j=3: trái = [1, 3, 8] + [7, 11, 18] | phải = [9, 15] + [19, 21, 25]
i=4, j=2: trái = [1, 3, 8, 9] + [7, 11] | phải = [15] + [18, 19, 21, 25]
i=5, j=1: trái = [1, 3, 8, 9, 15] + [7] | phải = [] + [11, 18, 19, 21, 25]


### Khi nào vách ngăn đúng?

Điều kiện 2 ("mọi phần tử trái `<=` mọi phần tử phải") nghe như phải so sánh rất nhiều cặp. Nhưng vì mỗi mảng đã sắp xếp:

- Trong cùng `nums1`: `nums1[i-1] <= nums1[i]` luôn đúng sẵn.
- Trong cùng `nums2`: `nums2[j-1] <= nums2[j]` luôn đúng sẵn.

Nên chỉ cần kiểm tra **2 cặp chéo**:

```python
maxLeftA <= minRightB    # nums1[i-1] <= nums2[j]
maxLeftB <= minRightA    # nums2[j-1] <= nums1[i]
```

Nhìn lại output cell trên với `A = [1, 3, 8, 9, 15]`, `B = [7, 11, 18, 19, 21, 25]`:

- `i = 2, j = 4`: `maxLeftB = 19 > minRightA = 8` → sai, nums2 đang "cho" quá nhiều sang trái.
- `i = 4, j = 2`: `maxLeftA = 9 <= minRightB = 18` và `maxLeftB = 11 <= minRightA = 15` → **đúng**!
  Nửa trái là `{1, 3, 8, 9, 7, 11}` → `max = 11` → median `= 11` (vì `total = 11` lẻ).

Kiểm tra lại: `sorted(A + B) = [1, 3, 7, 8, 9, 11, 15, 18, 19, 21, 25]`, phần tử giữa là `11`. Khớp.

### Vách ngăn ở mép: dùng ±inf

Nếu `i = 0`, bên trái vách của `nums1` **không có phần tử nào** → `nums1[i-1]` không tồn tại. Tương tự khi `i = m` thì `nums1[i]` không tồn tại. Với `nums2` cũng vậy khi `j = 0` hoặc `j = n`.

Mẹo: thay số không tồn tại bằng giá trị "trung lập":

- Bên trái trống → `maxLeft = -inf` (âm vô cùng): nhỏ hơn mọi số, nên điều kiện `maxLeft <= ...` luôn đúng và `max(...)` sẽ bỏ qua nó.
- Bên phải trống → `minRight = +inf` (dương vô cùng): lớn hơn mọi số, nên `... <= minRight` luôn đúng và `min(...)` bỏ qua nó.

Trong Python: `float('inf')` và `float('-inf')`.

Lưu ý quan trọng: **đừng** viết `nums1[i-1]` khi `i = 0`. Python không báo lỗi mà trả về `nums1[-1]` (phần tử **cuối** mảng) → sai âm thầm. Phải dùng `if i > 0 else float('-inf')`.

In [15]:
INF = float('inf')
print(-INF < -10**6, 10**6 < INF)          # True True
print(max(-INF, 5), min(INF, 5))           # 5 5
nums1 = [4, 9]
i = 0
print("nums1[i-1] khi i=0:", nums1[i - 1], "<- sai âm thầm (lấy phần tử cuối)!")
print("Cách đúng:", nums1[i - 1] if i > 0 else -INF)

True True
5 5
nums1[i-1] khi i=0: 9 <- sai âm thầm (lấy phần tử cuối)!
Cách đúng: -inf


### Binary search trên `i`: dịch vách sang trái hay sang phải?

Thử một `i` bất kỳ. Có đúng 3 khả năng:

1. Cả hai điều kiện đúng → **tìm thấy**.
2. `maxLeftA > minRightB`: `nums1` đang để **quá nhiều** phần tử (quá lớn) ở bên trái → phải **giảm `i`** → `hi = i - 1`.
3. `maxLeftB > minRightA`: `nums2` để quá nhiều sang trái, tức `nums1` để **quá ít** → **tăng `i`** → `lo = i + 1`.

Đây chính là binary search dạng "nhìn ở giữa → biết đáp án lệch về bên nào": khi `i` tăng, `maxLeftA` chỉ tăng còn `minRightB` chỉ giảm (vì `j` giảm), nên điều kiện có tính **đơn điệu**.

Vùng tìm kiếm ban đầu: `lo = 0`, `hi = m` (vách có thể đặt ở `m + 1` vị trí, từ trước phần tử đầu đến sau phần tử cuối). Chú ý `hi = m`, **không phải** `m - 1`.

### Vì sao chạy trên mảng **ngắn hơn**?

`j = half - i` phải nằm trong `[0, n]`. Nếu `m <= n` thì với mọi `i` trong `[0, m]`, `j` luôn hợp lệ. Nếu chạy trên mảng dài hơn, `j` có thể **âm** hoặc vượt `n`. Vì vậy đầu hàm ta đổi chỗ để `nums1` luôn là mảng ngắn hơn. Thêm lợi ích: thời gian là `O(log(min(m, n)))`.

In [16]:
long_arr, short_arr = [1, 2, 3, 4, 5, 6], [7]
m, n = len(long_arr), len(short_arr)
half = (m + n + 1) // 2
print("Nếu binary search trên mảng DÀI (m=6, n=1, half=4):")
print("  i:", list(range(m + 1)))
print("  j:", [half - i for i in range(m + 1)], "<- có j âm / j > n!")
m, n = n, m
print("Đổi chỗ, binary search trên mảng NGẮN (m=1, n=6):")
print("  i:", list(range(m + 1)), "-> j:", [half - i for i in range(m + 1)])

Nếu binary search trên mảng DÀI (m=6, n=1, half=4):
  i: [0, 1, 2, 3, 4, 5, 6]
  j: [4, 3, 2, 1, 0, -1, -2] <- có j âm / j > n!
Đổi chỗ, binary search trên mảng NGẮN (m=1, n=6):
  i: [0, 1] -> j: [4, 3]


### Tính median khi đã có vách đúng

- `total` lẻ: nửa trái có nhiều hơn 1 phần tử → median `= max(maxLeftA, maxLeftB)`.
- `total` chẵn: median `= (max(maxLeftA, maxLeftB) + min(minRightA, minRightB)) / 2`.

Ghép tất cả lại:

In [17]:
def median_binary_search(nums1, nums2):
    if len(nums1) > len(nums2):             # luôn binary search trên mảng ngắn hơn
        nums1, nums2 = nums2, nums1
    m, n = len(nums1), len(nums2)
    half = (m + n + 1) // 2                 # số phần tử của nửa trái
    INF = float('inf')

    lo, hi = 0, m                           # i có thể từ 0 đến m
    while lo <= hi:
        i = (lo + hi) // 2                  # vách trên nums1
        j = half - i                        # vách trên nums2

        maxLeftA = nums1[i - 1] if i > 0 else -INF
        minRightA = nums1[i] if i < m else INF
        maxLeftB = nums2[j - 1] if j > 0 else -INF
        minRightB = nums2[j] if j < n else INF

        if maxLeftA <= minRightB and maxLeftB <= minRightA:
            if (m + n) % 2 == 1:
                return float(max(maxLeftA, maxLeftB))
            return (max(maxLeftA, maxLeftB) + min(minRightA, minRightB)) / 2
        elif maxLeftA > minRightB:          # nums1 cho quá nhiều sang trái
            hi = i - 1
        else:                               # nums1 cho quá ít sang trái
            lo = i + 1

print(median_binary_search([1, 3], [2]))                          # 2.0
print(median_binary_search([1, 2], [3, 4]))                       # 2.5
print(median_binary_search([1, 3, 8, 9, 15], [7, 11, 18, 19, 21, 25]))  # 11.0

2.0
2.5
11.0


### Trace Cách 4

**Ca 1** — `nums1 = [1, 3, 8, 9, 15]`, `nums2 = [7, 11, 18, 19, 21, 25]` (`m = 5`, `n = 6`, `half = 6`, tổng lẻ):

| Vòng | `lo` | `hi` | `i` | `j` | `maxLeftA` | `minRightA` | `maxLeftB` | `minRightB` | Quyết định |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|---|
| 1 | 0 | 5 | 2 | 4 | 3 | 8 | 19 | 21 | `19 > 8` → `lo = 3` |
| 2 | 3 | 5 | 4 | 2 | 9 | 15 | 11 | 18 | đúng → `max(9, 11) = 11` |

**Ca 2** — `nums1 = [23, 26, 31, 35]`, `nums2 = [3, 5, 7, 9, 11, 16]` (`half = 5`, tổng chẵn). Ca này đi nhánh `hi = i - 1` và vách chạm mép trái (`-inf`):

| Vòng | `lo` | `hi` | `i` | `j` | `maxLeftA` | `minRightA` | `maxLeftB` | `minRightB` | Quyết định |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|---|
| 1 | 0 | 4 | 2 | 3 | 26 | 31 | 7 | 9 | `26 > 9` → `hi = 1` |
| 2 | 0 | 1 | 0 | 5 | -inf | 23 | 11 | 16 | đúng → `(11 + 16) / 2 = 13.5` |

**Ca 3** — Ví dụ 2 của đề, `nums1 = [1, 2]`, `nums2 = [3, 4]`: vách chạm mép phải của `nums1` (`+inf`) và mép trái của `nums2` (`-inf`).

In [18]:
def trace_partition(nums1, nums2):
    if len(nums1) > len(nums2):
        nums1, nums2 = nums2, nums1
    m, n = len(nums1), len(nums2)
    half = (m + n + 1) // 2
    INF = float('inf')
    lo, hi = 0, m
    print(f"nums1={nums1}, nums2={nums2}, half={half}")
    step = 1
    while lo <= hi:
        i = (lo + hi) // 2
        j = half - i
        mla = nums1[i - 1] if i > 0 else -INF
        mra = nums1[i] if i < m else INF
        mlb = nums2[j - 1] if j > 0 else -INF
        mrb = nums2[j] if j < n else INF
        print(f"Vòng {step}: lo={lo}, hi={hi}, i={i}, j={j} | "
              f"maxLeftA={mla}, minRightA={mra}, maxLeftB={mlb}, minRightB={mrb}")
        if mla <= mrb and mlb <= mra:
            if (m + n) % 2:
                ans = float(max(mla, mlb))
            else:
                ans = (max(mla, mlb) + min(mra, mrb)) / 2
            print("   vách đúng -> median =", ans)
            return ans
        elif mla > mrb:
            print(f"   {mla} > {mrb} -> dịch vách sang trái: hi = {i - 1}")
            hi = i - 1
        else:
            print(f"   {mlb} > {mra} -> dịch vách sang phải: lo = {i + 1}")
            lo = i + 1
        step += 1

trace_partition([1, 3, 8, 9, 15], [7, 11, 18, 19, 21, 25])
print()
trace_partition([23, 26, 31, 35], [3, 5, 7, 9, 11, 16])
print()
trace_partition([1, 2], [3, 4])

nums1=[1, 3, 8, 9, 15], nums2=[7, 11, 18, 19, 21, 25], half=6
Vòng 1: lo=0, hi=5, i=2, j=4 | maxLeftA=3, minRightA=8, maxLeftB=19, minRightB=21
   19 > 8 -> dịch vách sang phải: lo = 3
Vòng 2: lo=3, hi=5, i=4, j=2 | maxLeftA=9, minRightA=15, maxLeftB=11, minRightB=18
   vách đúng -> median = 11.0

nums1=[23, 26, 31, 35], nums2=[3, 5, 7, 9, 11, 16], half=5
Vòng 1: lo=0, hi=4, i=2, j=3 | maxLeftA=26, minRightA=31, maxLeftB=7, minRightB=9
   26 > 9 -> dịch vách sang trái: hi = 1
Vòng 2: lo=0, hi=1, i=0, j=5 | maxLeftA=-inf, minRightA=23, maxLeftB=11, minRightB=16
   vách đúng -> median = 13.5

nums1=[1, 2], nums2=[3, 4], half=2
Vòng 1: lo=0, hi=2, i=1, j=1 | maxLeftA=1, minRightA=2, maxLeftB=3, minRightB=4
   3 > 2 -> dịch vách sang phải: lo = 2
Vòng 2: lo=2, hi=2, i=2, j=0 | maxLeftA=2, minRightA=inf, maxLeftB=-inf, minRightB=3
   vách đúng -> median = 2.5


2.5

### Vì sao Cách 4 đúng?

**1. Luôn tồn tại một vách đúng.** Cắt mảng gộp (đã sắp xếp) ngay sau `half` phần tử đầu: những phần tử thuộc `nums1` trong đoạn đầu đó là một đoạn đầu của `nums1` (vì `nums1` tăng dần), tương tự với `nums2`. Gọi số phần tử của `nums1` là `i*` → vách `(i*, half - i*)` thỏa cả 2 điều kiện.

**2. Invariant (bất biến) của vòng lặp:** "nếu còn vách đúng thì nó nằm trong `[lo, hi]`".

- Nếu `maxLeftA > minRightB`: với mọi `i' >= i`, `maxLeftA` chỉ lớn hơn hoặc bằng, `minRightB` chỉ nhỏ hơn hoặc bằng → vẫn sai. Bỏ được `[i, hi]` → `hi = i - 1`.
- Nếu `maxLeftB > minRightA`: lập luận đối xứng, mọi `i' <= i` đều sai → `lo = i + 1`.

**3. Dừng:** mỗi vòng `[lo, hi]` co lại ít nhất 1. Vì vách đúng luôn tồn tại và không bao giờ bị loại, vòng lặp sẽ **return** trước khi `lo > hi`.

**4. Kết quả đúng:** khi 2 điều kiện chéo thỏa, nửa trái gồm đúng `half` phần tử nhỏ nhất, nên `max(nửa trái)` là phần tử thứ `half` và `min(nửa phải)` là phần tử thứ `half + 1` → đúng định nghĩa median.

### Độ phức tạp của Cách 4

- Thời gian: `O(log(min(m, n)))` — binary search trên `m + 1` vị trí của mảng ngắn hơn. Tốt hơn cả yêu cầu `O(log(m + n))`.
- Bộ nhớ: `O(1)`.

## 8. Code nộp LeetCode

Trên LeetCode, `List` đã được import sẵn. Trong notebook ta tự `from typing import List`.

Bản chính (Cách 4 — partition):

In [19]:
from typing import List

class Solution:
    def findMedianSortedArrays(self, nums1: List[int], nums2: List[int]) -> float:
        if len(nums1) > len(nums2):
            nums1, nums2 = nums2, nums1
        m, n = len(nums1), len(nums2)
        half = (m + n + 1) // 2
        INF = float('inf')

        lo, hi = 0, m
        while lo <= hi:
            i = (lo + hi) // 2
            j = half - i

            maxLeftA = nums1[i - 1] if i > 0 else -INF
            minRightA = nums1[i] if i < m else INF
            maxLeftB = nums2[j - 1] if j > 0 else -INF
            minRightB = nums2[j] if j < n else INF

            if maxLeftA <= minRightB and maxLeftB <= minRightA:
                if (m + n) % 2 == 1:
                    return float(max(maxLeftA, maxLeftB))
                return (max(maxLeftA, maxLeftB) + min(minRightA, minRightB)) / 2
            elif maxLeftA > minRightB:
                hi = i - 1
            else:
                lo = i + 1

Bản dự phòng (Cách 3 — phần tử thứ k), cũng đạt `O(log(m + n))`, hữu ích khi đề hỏi "phần tử thứ k" thay vì median:

In [20]:
class SolutionKth:
    def findMedianSortedArrays(self, nums1: List[int], nums2: List[int]) -> float:
        def kth(k: int) -> int:
            m, n = len(nums1), len(nums2)
            i = j = 0
            while True:
                if i == m:
                    return nums2[j + k - 1]
                if j == n:
                    return nums1[i + k - 1]
                if k == 1:
                    return min(nums1[i], nums2[j])
                half = k // 2
                ni = min(i + half, m) - 1
                nj = min(j + half, n) - 1
                if nums1[ni] <= nums2[nj]:
                    k -= ni - i + 1
                    i = ni + 1
                else:
                    k -= nj - j + 1
                    j = nj + 1

        total = len(nums1) + len(nums2)
        if total % 2 == 1:
            return float(kth(total // 2 + 1))
        return (kth(total // 2) + kth(total // 2 + 1)) / 2

### Bộ test

Các nhóm cần test:

1. Ví dụ của đề.
2. Một mảng rỗng (cả `nums1` rỗng và `nums2` rỗng).
3. Mỗi mảng 1 phần tử; tổng chỉ có 1 phần tử.
4. Hai mảng **không giao nhau** (mọi số của mảng này nhỏ hơn mảng kia) → vách chạm mép, dùng `±inf`.
5. Có phần tử **trùng nhau**, có số âm, giá trị biên `±10^6`.
6. Hai mảng chênh lệch độ dài lớn.

In [21]:
test_cases = [
    ([1, 3], [2], 2.0),
    ([1, 2], [3, 4], 2.5),
    ([], [1], 1.0),
    ([2], [], 2.0),
    ([], [2, 3], 2.5),
    ([1], [2], 1.5),
    ([1, 2, 3], [4, 5, 6], 3.5),
    ([4, 5, 6], [1, 2, 3], 3.5),
    ([1, 1, 1], [1, 1], 1.0),
    ([-5, -3, -1], [-2], -2.5),
    ([-10**6], [10**6], 0.0),
    ([1, 3, 8, 9, 15], [7, 11, 18, 19, 21, 25], 11.0),
    ([23, 26, 31, 35], [3, 5, 7, 9, 11, 16], 13.5),
    ([100], [1, 2, 3, 4, 5, 6, 7, 8], 5.0),
]

sol, sol_kth = Solution(), SolutionKth()
for nums1, nums2, expected in test_cases:
    got = sol.findMedianSortedArrays(nums1, nums2)
    got_kth = sol_kth.findMedianSortedArrays(nums1, nums2)
    assert got == expected, (nums1, nums2, got, expected)
    assert got_kth == expected, (nums1, nums2, got_kth, expected)
    assert isinstance(got, float)
print(f"Tất cả {len(test_cases)} test PASS cho cả Solution và SolutionKth!")

Tất cả 14 test PASS cho cả Solution và SolutionKth!


### Test đối chiếu chéo trên input ngẫu nhiên

Ta sinh nhiều cặp mảng ngẫu nhiên nhỏ (có trùng số, có mảng rỗng), rồi so **tất cả các cách** với brute force `median_sort`. Dùng `random.seed(...)` cố định để lần chạy nào cũng ra cùng dữ liệu.

Mẹo: hai lời giải tính ra cùng một phép chia nên so sánh `==` với `float` ở đây là an toàn (cùng là `(a + b) / 2` hoặc `float(x)` với `a`, `b`, `x` là số nguyên).

In [22]:
import random

random.seed(2024)
solvers = {
    "median_merge": median_merge,
    "median_merge_half": median_merge_half,
    "median_kth": median_kth,
    "median_binary_search": median_binary_search,
    "Solution": Solution().findMedianSortedArrays,
    "SolutionKth": SolutionKth().findMedianSortedArrays,
}

trials = 3000
for _ in range(trials):
    m = random.randint(0, 8)
    n = random.randint(0 if m > 0 else 1, 8)
    nums1 = sorted(random.randint(-10, 10) for _ in range(m))
    nums2 = sorted(random.randint(-10, 10) for _ in range(n))
    expected = median_sort(nums1, nums2)
    for name, f in solvers.items():
        got = f(nums1, nums2)
        assert got == expected, (name, nums1, nums2, got, expected)

print(f"{trials} bộ ngẫu nhiên: {len(solvers)} cách đều khớp với median_sort.")

3000 bộ ngẫu nhiên: 6 cách đều khớp với median_sort.


## 9. So sánh các cách

Đặt `m = len(nums1)`, `n = len(nums2)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Đạt yêu cầu đề? | Khi nào dùng |
|---|---|---|---|:---:|---|
| 1. Gộp + sort | Đúng định nghĩa | `O((m+n) log(m+n))` | `O(m+n)` | Không | Brute force để kiểm tra |
| 2a. Merge hai con trỏ | Tận dụng đã sắp xếp | `O(m+n)` | `O(m+n)` | Không | Hiểu bài, bài Merge Sorted Array |
| 2b. Merge chỉ tới giữa | Giữ `prev`, `cur` | `O(m+n)` | `O(1)` | Không | Phỏng vấn hỏi cách đơn giản trước |
| 3. Phần tử thứ k | Loại `k/2` mỗi bước | `O(log(m+n))` | `O(1)` | Có | Đề hỏi phần tử thứ k bất kỳ |
| 4. Partition | Binary search vách ngăn | `O(log(min(m,n)))` | `O(1)` | Có | **Cách chính để nộp** |

Ghi chú thực tế: với `m + n <= 2000`, cả Cách 1 cũng chạy rất nhanh trên LeetCode. Nhưng bài này nổi tiếng chính vì yêu cầu `O(log(m+n))` — trong phỏng vấn, người ta muốn thấy Cách 3 hoặc Cách 4.

## 10. Các lỗi hay gặp

### Lỗi 1: `hi = m - 1` thay vì `hi = m`

Vách trên `nums1` có `m + 1` vị trí (`i = 0, 1, ..., m`). Nếu đặt `hi = m - 1` thì không bao giờ thử `i = m` (lấy **hết** `nums1` sang trái). Với Ví dụ 2 (`[1, 2]`, `[3, 4]`), vách đúng chính là `i = 2 = m` → vòng lặp kết thúc mà không `return`, hàm trả về `None`.

**Sửa:** `lo, hi = 0, m`.

### Lỗi 2: Không đổi chỗ để chạy trên mảng ngắn hơn

`j = half - i` có thể âm. Python không báo lỗi khi index âm (`nums2[-1]` là phần tử cuối), nên kết quả **sai âm thầm**, hoặc `IndexError` khi `j > n`.

**Sửa:** `if len(nums1) > len(nums2): nums1, nums2 = nums2, nums1`.

### Lỗi 3: Đọc `nums1[i-1]` khi `i = 0` mà không dùng `-inf`

Giống lỗi 2: `nums1[-1]` là phần tử cuối, không phải "không có gì".

**Sửa:** `maxLeftA = nums1[i - 1] if i > 0 else float('-inf')`, tương tự cho 3 giá trị còn lại.

### Lỗi 4: Dùng `//` khi tính median chẵn

`(2 + 3) // 2 = 2` chứ không phải `2.5`. **Sửa:** dùng `/`.

### Lỗi 5: Công thức `half` sai cho tổng lẻ

Nếu dùng `half = (m + n) // 2` thì với tổng lẻ, phần tử giữa nằm ở **nửa phải**, lúc đó phải trả `min(minRight...)` chứ không phải `max(maxLeft...)`. Trộn hai quy ước là sai. **Sửa:** chọn một quy ước và nhất quán; notebook này dùng `half = (m + n + 1) // 2` và trả `max(maxLeft)` khi lẻ.

Cell dưới chạy thử Lỗi 1, Lỗi 4 và Lỗi 5 để thấy tận mắt:

In [23]:
def median_wrong_hi(nums1, nums2):
    if len(nums1) > len(nums2):
        nums1, nums2 = nums2, nums1
    m, n = len(nums1), len(nums2)
    half = (m + n + 1) // 2
    INF = float('inf')
    lo, hi = 0, m - 1                       # LỖI: thiếu vị trí i = m
    while lo <= hi:
        i = (lo + hi) // 2
        j = half - i
        mla = nums1[i - 1] if i > 0 else -INF
        mra = nums1[i] if i < m else INF
        mlb = nums2[j - 1] if j > 0 else -INF
        mrb = nums2[j] if j < n else INF
        if mla <= mrb and mlb <= mra:
            if (m + n) % 2:
                return float(max(mla, mlb))
            return (max(mla, mlb) + min(mra, mrb)) / 2
        elif mla > mrb:
            hi = i - 1
        else:
            lo = i + 1

print("Lỗi 1 - hi = m - 1:", median_wrong_hi([1, 2], [3, 4]), "| đúng:", median_binary_search([1, 2], [3, 4]))
print("Lỗi 4 - dùng //   :", (2 + 3) // 2, "| đúng:", (2 + 3) / 2)
# Lỗi 5: half = (m+n)//2 nhưng vẫn trả max(nửa trái) khi lẻ
arr = [1, 2, 3, 4, 5]
wrong_half = len(arr) // 2
print("Lỗi 5 - half sai  :", max(arr[:wrong_half]), "| đúng:", max(arr[:(len(arr) + 1) // 2]))

Lỗi 1 - hi = m - 1: None | đúng: 2.5
Lỗi 4 - dùng //   : 2 | đúng: 2.5
Lỗi 5 - half sai  : 2 | đúng: 3


## 11. Mẹo nhận dạng pattern

- Thấy **"mảng đã sắp xếp"** + yêu cầu **`O(log ...)`** → gần như chắc chắn là **binary search**.
- Binary search không nhất thiết tìm một **giá trị**; nó có thể tìm một **vị trí cắt** (vách ngăn), một **chỉ số chuyển trạng thái**, hoặc một **đáp án** trên miền giá trị. Hãy hỏi: "nhìn ở giữa, tôi có biết đáp án nằm bên nào không?"
- **Median / phần tử thứ k** trên dữ liệu đã sắp xếp → nghĩ tới "nửa trái có `half` phần tử, mọi phần tử trái `<=` mọi phần tử phải".
- Có hai biến ràng buộc nhau (`i + j = half`) → **chỉ binary search trên một biến**, biến còn lại tự suy ra. Chọn biến có miền nhỏ hơn.
- Vách ngăn có thể chạm mép → dùng **lính canh (sentinel)** `±inf` thay vì viết nhiều `if` đặc biệt.

## 12. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 704 — Binary Search | Binary search cơ bản (mục 5) |
| LeetCode 35 — Search Insert Position | Chính là lower bound |
| LeetCode 88 — Merge Sorted Array | Merge hai con trỏ (Cách 2) |
| LeetCode 295 — Find Median from Data Stream | Median khi dữ liệu đến dần (dùng 2 heap) |
| LeetCode 480 — Sliding Window Median | Median trên cửa sổ trượt |
| LeetCode 378 — Kth Smallest Element in a Sorted Matrix | Phần tử thứ k, binary search trên giá trị |
| Phần tử thứ k của hai mảng đã sắp xếp | Dùng nguyên `kth_smallest` (Cách 3), hoặc partition với `half = k` |

## 13. Bài tập vận dụng

Hãy tự làm trước, chỉ xem **Gợi ý** khi bí, và xem **Đáp án** ở cuối notebook sau cùng.

### Bài 1 — Đoán output (không chạy code)

Đoán kết quả từng dòng:

```python
print(len([1, 2, 3, 4, 5, 6]) // 2)
print((7 + 1) // 2, (8 + 1) // 2)
print((4 + 7) / 2, (4 + 7) // 2)
print(max(float('-inf'), -3), min(float('inf'), 10**6))
nums = [5, 8, 13]
print(nums[0 - 1])
```

### Bài 2 — Tính tay vách ngăn

Cho `nums1 = [2, 5, 9]`, `nums2 = [1, 4, 6, 10]`.

1. Tính `m`, `n`, `half`.
2. Chạy tay Cách 4: mỗi vòng ghi `lo`, `hi`, `i`, `j`, `maxLeftA`, `minRightA`, `maxLeftB`, `minRightB` và quyết định.
3. Median bằng bao nhiêu? Kiểm tra lại bằng cách gộp.

### Bài 3 — Điền TODO cho Cách 4

Hoàn thiện hàm theo các TODO. Không nhìn lại mục 7 nếu có thể.

In [24]:
def exercise_median_partition(nums1, nums2):
    # TODO 1: đổi chỗ để nums1 là mảng ngắn hơn
    # TODO 2: tính m, n, half = (m + n + 1) // 2
    # TODO 3: lo, hi = 0, m   (chú ý: hi = m)
    # TODO 4: while lo <= hi: tính i, j = half - i
    # TODO 5: tính maxLeftA, minRightA, maxLeftB, minRightB (dùng ±inf ở mép)
    # TODO 6: nếu vách đúng -> trả median (lẻ / chẵn)
    # TODO 7: nếu maxLeftA > minRightB -> hi = i - 1, ngược lại lo = i + 1
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_median_partition([1, 3], [2]) == 2.0
# assert exercise_median_partition([1, 2], [3, 4]) == 2.5
# assert exercise_median_partition([], [1]) == 1.0
# assert exercise_median_partition([2, 5, 9], [1, 4, 6, 10]) == 5.0

### Bài 4 — Tự viết: phần tử nhỏ thứ k của hai mảng

Viết `exercise_kth(nums1, nums2, k)` trả về phần tử nhỏ thứ `k` (`k` đếm từ 1, `1 <= k <= m + n`) trong `O(log(m + n))`. Được chọn cách loại `k/2` (Cách 3) **hoặc** cách partition với nửa trái có đúng `k` phần tử.

In [25]:
def exercise_kth(nums1, nums2, k):
    # TODO: không được gộp rồi sort
    raise NotImplementedError

# Gợi ý test:
# assert exercise_kth([1, 3, 5, 7], [2, 4, 6, 8, 10], 5) == 5
# assert exercise_kth([], [3, 4], 2) == 4
# assert exercise_kth([1, 2], [3], 1) == 1
# assert exercise_kth([1, 2], [3], 3) == 3

### Bài 5 — Biến thể: hai mảng sắp xếp **giảm dần**

Lần này `nums1` và `nums2` đều sắp xếp **giảm dần** (ví dụ `[9, 5, 2]`). Viết `exercise_median_desc` trong `O(log(min(m, n)))`.

Yêu cầu: **không** đảo ngược mảng (đảo tốn `O(m + n)`) và không sort lại. Hãy suy nghĩ: vách ngăn bây giờ chia "nửa lớn" và "nửa nhỏ"; điều kiện chéo và hướng dịch vách thay đổi thế nào?

In [26]:
def exercise_median_desc(nums1, nums2):
    # TODO: binary search vách ngăn trên mảng ngắn hơn, nhưng với thứ tự giảm dần
    pass

# Gợi ý test:
# assert exercise_median_desc([3, 1], [2]) == 2.0
# assert exercise_median_desc([2, 1], [4, 3]) == 2.5
# assert exercise_median_desc([], [9, 5, 2]) == 5.0
# assert exercise_median_desc([15, 9, 8, 3, 1], [25, 21, 19, 18, 11, 7]) == 11.0

### Bài 6 — Mini challenge: binary search trên **giá trị**

Một góc nhìn khác: thay vì binary search trên **vị trí**, hãy binary search trên **giá trị** `x` trong khoảng `[-10^6, 10^6]`.

- Gọi `count_leq(x)` = số phần tử `<= x` trong cả hai mảng. Tính bằng `bisect.bisect_right` trên từng mảng → `O(log m + log n)`.
- `count_leq(x)` **tăng dần** theo `x` (đơn điệu!). Phần tử thứ `k` là **giá trị `x` nhỏ nhất** sao cho `count_leq(x) >= k`.

Viết `exercise_median_by_value(nums1, nums2)` dùng ý tưởng này. Độ phức tạp là bao nhiêu? (Lưu ý: cách này phụ thuộc vào việc giá trị là **số nguyên** trong miền giới hạn.)

In [27]:
def exercise_median_by_value(nums1, nums2):
    # TODO 1: viết count_leq(x) bằng bisect.bisect_right
    # TODO 2: viết kth_by_value(k): binary search x trong [-10**6, 10**6]
    # TODO 3: dùng kth_by_value để tính median (lẻ / chẵn)
    pass

# Gợi ý test:
# assert exercise_median_by_value([1, 3], [2]) == 2.0
# assert exercise_median_by_value([1, 2], [3, 4]) == 2.5
# assert exercise_median_by_value([-10**6], [10**6]) == 0.0

### Gợi ý theo mức

**Bài 2**
- Mức 1: `nums1` đã ngắn hơn, `half = (3 + 4 + 1) // 2`.
- Mức 2: vòng đầu `i = (0 + 3) // 2`.
- Mức 3: vòng 1 `i = 1, j = 3`; so `maxLeftB = 6` với `minRightA = 5`.

**Bài 3**
- Mức 1: chép lại khung "chọn `i` → suy ra `j` → kiểm tra 2 cặp chéo".
- Mức 2: `x if điều_kiện else -INF` cho từng giá trị sát vách.
- Mức 3: điều kiện đúng là `maxLeftA <= minRightB and maxLeftB <= minRightA`.

**Bài 4**
- Mức 1: dùng lại Cách 3 với offset `i`, `j`.
- Mức 2: nếu dùng partition: nửa trái có `k` phần tử, `i` chạy trong `[max(0, k - n), min(k, m)]`.
- Mức 3: với partition, đáp án là `max(maxLeftA, maxLeftB)`.

**Bài 5**
- Mức 1: trong mảng giảm dần, "nửa trái" chứa các số **lớn**.
- Mức 2: điều kiện đúng thành `minLeftA >= maxRightB` và `minLeftB >= maxRightA`; ở mép trái dùng `+inf`, mép phải dùng `-inf`.
- Mức 3: khi lẻ, median `= min(minLeftA, minLeftB)`; nếu `minLeftA < maxRightB` thì `nums1` cho quá nhiều sang trái → `hi = i - 1`.

**Bài 6**
- Mức 1: đây là dạng "tìm vị trí đầu tiên điều kiện đúng" (lower bound) ở mục 5, nhưng trên trục giá trị.
- Mức 2: `lo, hi = -10**6, 10**6`; `while lo < hi`: nếu `count_leq(mid) >= k` thì `hi = mid` else `lo = mid + 1`.
- Mức 3: cẩn thận `(lo + hi) // 2` với số âm trong Python làm tròn xuống, vẫn đúng với khuôn `hi = mid` / `lo = mid + 1`.

## 14. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

```text
3
4 4
5.5 5
-3 1000000
13
```

Dòng cuối: `nums[0 - 1]` là `nums[-1]` → phần tử **cuối** `13`. Đây là lý do phải dùng `-inf` khi `i = 0`.

### Đáp án Bài 2

`m = 3`, `n = 4`, `half = (3 + 4 + 1) // 2 = 4`, tổng `7` lẻ.

| Vòng | `lo` | `hi` | `i` | `j` | `maxLeftA` | `minRightA` | `maxLeftB` | `minRightB` | Quyết định |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|---|
| 1 | 0 | 3 | 1 | 3 | 2 | 5 | 6 | 10 | `6 > 5` → `lo = 2` |
| 2 | 2 | 3 | 2 | 2 | 5 | 9 | 4 | 6 | đúng → `max(5, 4) = 5` |

Gộp: `[1, 2, 4, 5, 6, 9, 10]` → phần tử giữa là `5`. Cell dưới kiểm chứng bằng `trace_partition`.

In [28]:
print(len([1, 2, 3, 4, 5, 6]) // 2)
print((7 + 1) // 2, (8 + 1) // 2)
print((4 + 7) / 2, (4 + 7) // 2)
print(max(float('-inf'), -3), min(float('inf'), 10**6))
nums = [5, 8, 13]
print(nums[0 - 1])
print()
trace_partition([2, 5, 9], [1, 4, 6, 10])

3


4 4
5.5 5
-3 1000000
13

nums1=[2, 5, 9], nums2=[1, 4, 6, 10], half=4
Vòng 1: lo=0, hi=3, i=1, j=3 | maxLeftA=2, minRightA=5, maxLeftB=6, minRightB=10
   6 > 5 -> dịch vách sang phải: lo = 2
Vòng 2: lo=2, hi=3, i=2, j=2 | maxLeftA=5, minRightA=9, maxLeftB=4, minRightB=6
   vách đúng -> median = 5.0


5.0

In [29]:
import bisect

def answer_median_partition(nums1, nums2):
    if len(nums1) > len(nums2):
        nums1, nums2 = nums2, nums1
    m, n = len(nums1), len(nums2)
    half = (m + n + 1) // 2
    INF = float('inf')
    lo, hi = 0, m
    while lo <= hi:
        i = (lo + hi) // 2
        j = half - i
        maxLeftA = nums1[i - 1] if i > 0 else -INF
        minRightA = nums1[i] if i < m else INF
        maxLeftB = nums2[j - 1] if j > 0 else -INF
        minRightB = nums2[j] if j < n else INF
        if maxLeftA <= minRightB and maxLeftB <= minRightA:
            if (m + n) % 2 == 1:
                return float(max(maxLeftA, maxLeftB))
            return (max(maxLeftA, maxLeftB) + min(minRightA, minRightB)) / 2
        elif maxLeftA > minRightB:
            hi = i - 1
        else:
            lo = i + 1


def answer_kth(nums1, nums2, k):
    # Cách partition: nửa trái có đúng k phần tử
    if len(nums1) > len(nums2):
        nums1, nums2 = nums2, nums1
    m, n = len(nums1), len(nums2)
    INF = float('inf')
    lo, hi = max(0, k - n), min(k, m)       # để j = k - i nằm trong [0, n]
    while lo <= hi:
        i = (lo + hi) // 2
        j = k - i
        maxLeftA = nums1[i - 1] if i > 0 else -INF
        minRightA = nums1[i] if i < m else INF
        maxLeftB = nums2[j - 1] if j > 0 else -INF
        minRightB = nums2[j] if j < n else INF
        if maxLeftA <= minRightB and maxLeftB <= minRightA:
            return max(maxLeftA, maxLeftB)
        elif maxLeftA > minRightB:
            hi = i - 1
        else:
            lo = i + 1


def answer_median_desc(nums1, nums2):
    if len(nums1) > len(nums2):
        nums1, nums2 = nums2, nums1
    m, n = len(nums1), len(nums2)
    half = (m + n + 1) // 2
    INF = float('inf')
    lo, hi = 0, m
    while lo <= hi:
        i = (lo + hi) // 2
        j = half - i
        minLeftA = nums1[i - 1] if i > 0 else INF     # nửa trái = các số lớn
        maxRightA = nums1[i] if i < m else -INF
        minLeftB = nums2[j - 1] if j > 0 else INF
        maxRightB = nums2[j] if j < n else -INF
        if minLeftA >= maxRightB and minLeftB >= maxRightA:
            if (m + n) % 2 == 1:
                return float(min(minLeftA, minLeftB))
            return (min(minLeftA, minLeftB) + max(maxRightA, maxRightB)) / 2
        elif minLeftA < maxRightB:                   # nums1 cho quá nhiều sang trái
            hi = i - 1
        else:
            lo = i + 1


def answer_median_by_value(nums1, nums2):
    def count_leq(x):
        return bisect.bisect_right(nums1, x) + bisect.bisect_right(nums2, x)

    def kth_by_value(k):
        lo, hi = -10**6, 10**6
        while lo < hi:
            mid = (lo + hi) // 2
            if count_leq(mid) >= k:
                hi = mid
            else:
                lo = mid + 1
        return lo

    total = len(nums1) + len(nums2)
    if total % 2 == 1:
        return float(kth_by_value(total // 2 + 1))
    return (kth_by_value(total // 2) + kth_by_value(total // 2 + 1)) / 2


# Test theo đề bài tập
assert answer_median_partition([1, 3], [2]) == 2.0
assert answer_median_partition([1, 2], [3, 4]) == 2.5
assert answer_median_partition([], [1]) == 1.0
assert answer_median_partition([2, 5, 9], [1, 4, 6, 10]) == 5.0
assert answer_kth([1, 3, 5, 7], [2, 4, 6, 8, 10], 5) == 5
assert answer_kth([], [3, 4], 2) == 4
assert answer_kth([1, 2], [3], 1) == 1
assert answer_kth([1, 2], [3], 3) == 3
assert answer_median_desc([3, 1], [2]) == 2.0
assert answer_median_desc([2, 1], [4, 3]) == 2.5
assert answer_median_desc([], [9, 5, 2]) == 5.0
assert answer_median_desc([15, 9, 8, 3, 1], [25, 21, 19, 18, 11, 7]) == 11.0
assert answer_median_by_value([1, 3], [2]) == 2.0
assert answer_median_by_value([1, 2], [3, 4]) == 2.5
assert answer_median_by_value([-10**6], [10**6]) == 0.0

# Đối chiếu ngẫu nhiên với brute force
random.seed(7)
for _ in range(1500):
    m = random.randint(0, 7)
    n = random.randint(0 if m > 0 else 1, 7)
    a = sorted(random.randint(-20, 20) for _ in range(m))
    b = sorted(random.randint(-20, 20) for _ in range(n))
    exp = median_sort(a, b)
    assert answer_median_partition(a, b) == exp
    assert answer_median_by_value(a, b) == exp
    assert answer_median_desc(a[::-1], b[::-1]) == exp
    k = random.randint(1, m + n)
    assert answer_kth(a, b, k) == sorted(a + b)[k - 1]

print("Đáp án tham khảo chạy PASS toàn bộ test!")

Đáp án tham khảo chạy PASS toàn bộ test!


**Độ phức tạp Bài 6:** binary search trên miền giá trị có `R = 2·10^6 + 1` khả năng → khoảng `log₂ R ≈ 21` vòng; mỗi vòng `count_leq` tốn `O(log m + log n)`. Tổng: `O(log R · (log m + log n))`. Rất nhanh, nhưng phụ thuộc miền giá trị và chỉ hợp với số nguyên; Cách 4 vẫn gọn và tổng quát hơn.

## 15. Template cuối cùng nên nhớ

Khung "binary search vách ngăn" trong 5 dòng ý tưởng:

1. Đổi chỗ để `nums1` ngắn hơn. `half = (m + n + 1) // 2`.
2. `lo, hi = 0, m`. Lặp `while lo <= hi`: `i = (lo + hi) // 2`, `j = half - i`.
3. Lấy 4 số sát vách, mép thì dùng `±inf`.
4. Đúng khi `maxLeftA <= minRightB and maxLeftB <= minRightA` → lẻ: `max(trái)`; chẵn: `(max(trái) + min(phải)) / 2`.
5. `maxLeftA > minRightB` → `hi = i - 1`; ngược lại `lo = i + 1`.

```python
class Solution:
    def findMedianSortedArrays(self, nums1: List[int], nums2: List[int]) -> float:
        if len(nums1) > len(nums2):
            nums1, nums2 = nums2, nums1
        m, n = len(nums1), len(nums2)
        half = (m + n + 1) // 2
        INF = float('inf')
        lo, hi = 0, m
        while lo <= hi:
            i = (lo + hi) // 2
            j = half - i
            maxLeftA = nums1[i - 1] if i > 0 else -INF
            minRightA = nums1[i] if i < m else INF
            maxLeftB = nums2[j - 1] if j > 0 else -INF
            minRightB = nums2[j] if j < n else INF
            if maxLeftA <= minRightB and maxLeftB <= minRightA:
                if (m + n) % 2 == 1:
                    return float(max(maxLeftA, maxLeftB))
                return (max(maxLeftA, maxLeftB) + min(minRightA, minRightB)) / 2
            elif maxLeftA > minRightB:
                hi = i - 1
            else:
                lo = i + 1
```

## 16. Checklist tự kiểm tra trước khi submit

- Mình đã đổi chỗ để binary search trên **mảng ngắn hơn** chưa?
- `hi` khởi tạo là `m` (không phải `m - 1`) chưa?
- `half = (m + n + 1) // 2` và khi lẻ trả `max(maxLeftA, maxLeftB)` — có nhất quán không?
- Mọi chỗ đọc `nums[i-1]`, `nums[i]`, `nums[j-1]`, `nums[j]` đều có kiểm tra mép và dùng `±inf` chưa?
- Hướng dịch vách: `maxLeftA > minRightB` → giảm `i` (`hi = i - 1`), còn lại → tăng `i` (`lo = i + 1`)?
- Median chẵn dùng `/ 2` (không phải `// 2`), kết quả trả về là `float`?
- Đã thử: một mảng rỗng, hai mảng không giao nhau, phần tử trùng, số âm, độ dài chênh lệch lớn?